# Auliya Discriminative Representation Audit v1

Frozen B → C pilot, **no detector training, no test A**. Add the exact pilot bundle and RGB last.pt as Kaggle Input. Enable Internet and a GPU (for feature stage). Save Version to retain outputs. See the protocol in the pinned repository.

In [ ]:
# Configuration
SCIENTIFIC_CODE_COMMIT = "962913d9ef8b97ff678db515dfbf7713e1ea908b"
RUN_VISUAL = True
RUN_FEATURES = True
DEVICE = "cuda:0"  # features only; "cpu" is allowed but slower
BUNDLE_OVERRIDE = ""  # optional exact path under /kaggle/input
CHECKPOINT_OVERRIDE = ""  # optional exact path under /kaggle/input
PILOT_SHA = "b8795e189b85a13201e54a562cbe0f42fa7e0f2bfc0eb4b6d04f85d2b4cf57be"
RGB_SHA = "85b9b459fcd5079de3c27f8b897a4bd0c7160efa96bea25957c5eadd4d617ed3"

import os, hashlib, json, shutil, subprocess, sys
from pathlib import Path
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "auliya-discriminative-representation-v1"
REPO = WORK / "coffee-bean-detection-audit"
assert INPUT.is_dir() and WORK.is_dir(), "Run in Kaggle"
assert RUN_VISUAL or RUN_FEATURES, "Enable at least one stage"

def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_exact(filename, expected, override=""):
    suffix = Path(filename).suffix.lower()
    if override:
        candidate = Path(override).resolve()
        if not candidate.is_relative_to(INPUT.resolve()) or not candidate.is_file():
            raise RuntimeError(f"Override must point to an existing file under {INPUT}: {candidate}")
        candidates = [candidate]
    else:
        # Prefer the canonical name, but accept renamed byte-identical inputs.
        candidates = sorted(
            (p for p in INPUT.rglob("*") if p.is_file() and p.suffix.lower() == suffix),
            key=lambda p: (p.name != filename, str(p)),
        )
    print(f"Searching for {filename}: {len(candidates)} {suffix} candidate(s)", flush=True)
    good = []
    for p in candidates:
        digest = file_sha(p)
        print(f"  {p} | {p.stat().st_size} bytes | SHA256={digest}", flush=True)
        if digest == expected:
            good.append(p)
    if not good:
        print("Mounted input directories:", sorted(str(p) for p in INPUT.iterdir()), flush=True)
        if suffix == ".zip":
            contracts = list(INPUT.rglob("development_contract.json"))
            if contracts:
                print("Extracted development folders found:", [str(p) for p in contracts[:10]], flush=True)
                print("An extracted/repacked folder cannot be verified against the original ZIP SHA.", flush=True)
        raise RuntimeError(
            f"No verified {filename} found (expected SHA256 {expected}). "
            "Attach the unchanged original file through Add Input / private dataset. "
            "Do not re-zip or substitute another checkpoint. "
            "See the candidate paths and hashes above."
        )
    # Duplicate copies with the same SHA are byte-identical. Prefer canonical path.
    if len(good) > 1:
        print("Multiple byte-identical copies found; using:", good[0], flush=True)
    return good[0]

BUNDLE = find_exact("auliya-robusta-sni-frequency-colab-v1.zip", PILOT_SHA, BUNDLE_OVERRIDE)
CHECKPOINT = find_exact("last.pt", RGB_SHA, CHECKPOINT_OVERRIDE) if RUN_FEATURES else None
print("Verified B/C bundle:", BUNDLE)
print("Verified RGB checkpoint:", CHECKPOINT)

prior = sorted(p for p in INPUT.rglob(PROJECT.name) if p.is_dir() and (p/"logs").is_dir())
if len(prior)>1:
    raise RuntimeError("Ambiguous prior Notebook Output; attach only one")
if prior:
    if PROJECT.exists() and any(PROJECT.iterdir()):
        print("Local working output already exists; not overwriting it")
    else:
        shutil.copytree(prior[0],PROJECT,dirs_exist_ok=True)
        print("Restored:",prior[0])
PROJECT.mkdir(exist_ok=True)
(PROJECT/"logs").mkdir(exist_ok=True)

def run(cmd, log=None, cwd=None):
    cmd=list(map(str,cmd))
    print("$ "+" ".join(cmd),flush=True)
    if log is None:
        subprocess.run(cmd,check=True,cwd=cwd)
        return
    with Path(log).open("a",encoding="utf-8") as stream:
        with subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True)
                stream.write(line)
                stream.flush()
            if p.wait()!=0:
                raise RuntimeError(f"Command failed; see {log}. Completed image shards retained.")


In [ ]:
# Immutable science checkout; preserve existing experiment output.
if not REPO.exists():
    run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-detection.git",REPO])
run(["git","-C",REPO,"checkout","--quiet","--detach",SCIENTIFIC_CODE_COMMIT])
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
assert actual==SCIENTIFIC_CODE_COMMIT, actual
# Keep Kaggle's torch/CUDA stack; install pinned Ultralytics only if needed.
run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.96","scikit-learn","pytest"])
run([sys.executable,"-m","pip","install","-q","--no-deps","-e",REPO])
sys.path.insert(0,str(REPO/"src"))
os.environ["PYTHONPATH"]=str(REPO/"src")+os.pathsep+os.environ.get("PYTHONPATH","")

import torch, ultralytics, numpy, cv2, sklearn
assert ultralytics.__version__=="8.4.96"
if RUN_FEATURES and DEVICE.startswith("cuda"):
    assert torch.cuda.is_available(), "Enable Kaggle GPU"
env={"commit":actual,"torch":torch.__version__,"ultralytics":ultralytics.__version__,
     "numpy":numpy.__version__,"opencv":cv2.__version__,"sklearn":sklearn.__version__,
     "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(json.dumps(env,indent=2))
env_path=PROJECT/"environment.json"
if env_path.exists() and json.loads(env_path.read_text())!=env:
    raise RuntimeError("Resume environment changed; use a fresh output")
env_path.write_text(json.dumps(env,indent=2))
run([sys.executable,"-m","pip","freeze"],log=PROJECT/"logs/pip-freeze.txt")
run([sys.executable,"-m","pytest","-q","tests/test_auliya_representation_audit.py"],
    cwd=REPO,log=PROJECT/"logs/tests.log")


In [ ]:
# Only fit lightweight sklearn probes, never train YOLO.
for stage,enabled in [("visual",RUN_VISUAL),("features",RUN_FEATURES)]:
    if not enabled:
        continue
    cmd=[sys.executable,"-u","-m","coffee_detector.analysis.auliya_representation_audit",
         "--bundle",BUNDLE,"--stage",stage,"--output",PROJECT/stage,
         "--device",DEVICE if stage=="features" else "cpu"]
    if stage=="features":
        cmd.extend(["--checkpoint",CHECKPOINT])
    run(cmd,cwd=REPO,log=PROJECT/"logs"/f"{stage}.log")


In [ ]:
# Results for mechanistic review; no automatic architecture selection.
for stage in ("visual","features"):
    p=PROJECT/stage/"report.json"
    if not p.exists():
        continue
    report=json.loads(p.read_text())
    print("\n",stage.upper(),report["decision"])
    for name,row in report["results"].items():
        if "macro18" in row:
            print(f"{name:22s} Macro18={row['macro18']:.4f}  Hard5={row['hard5']:.4f}  Worst18={row['worst18']:.4f}")
    if "raw_anchor_coverage" in report["results"]:
        print(report["results"]["raw_anchor_coverage"])
    print("Report:",p)
    print("Predictions:",PROJECT/stage/"validation_predictions.csv")
print("Save Version to retain the complete project directory:",PROJECT)
